In [19]:
import scipy.io
import numpy as np
from pydub import AudioSegment

In [20]:
# load the .mat file
file_path = r'E:\HRTF\HRTF_code\cipic-hrtf-database-master\standard_hrir_database\subject_003\hrir_final.mat'
data = scipy.io.loadmat(file_path)

print(data.keys())
left_ear = data['hrir_l'] 
right_ear = data['hrir_r']

# print the shapes of the left and right ear data
print("azimuth,elevation,hrir_l", left_ear.shape)
print("azimuth,elevation,hrir_r", right_ear.shape)


dict_keys(['__header__', '__version__', '__globals__', 'OnR', 'OnL', 'ITD', 'hrir_r', 'hrir_l', 'name'])
azimuth,elevation,hrir_l (25, 50, 200)
azimuth,elevation,hrir_r (25, 50, 200)


In [21]:
# load the audio
audio_file=r'sample_audio.wav'
audio = AudioSegment.from_file(audio_file)

#covnert to mono
audio = audio.set_channels(1) 

#normalise audio
audio = audio.normalize()

# check the sample rate of the audio
if audio.frame_rate != 44100:
    audio = audio.set_frame_rate(44100)

samples = np.array(audio.get_array_of_samples())

In [22]:
# select a specific azimuth and elevation
azimuth_index = 12
elevation_index = 1

# get the corresponding HRIR for the left and right ear
hrir_left = left_ear[azimuth_index, elevation_index, :]
hrir_right = right_ear[azimuth_index, elevation_index, :]

In [23]:
# Apply the convolution to get the binaural audio

test_audio_left = np.convolve(samples, hrir_left)
test_audio_right = np.convolve(samples, hrir_right)

In [24]:
# combine and generate the stereo audio
stereo_audio = np.stack((test_audio_left, test_audio_right), axis=1)

# normalize the stereo audio
stereo_audio = stereo_audio / np.max(np.abs(stereo_audio))

# Scale to int16 range and cast
stereo_audio_int16 = (stereo_audio * 32767).astype(np.int16)

# Create pydub AudioSegment from raw PCM bytes
stereo_audio_segment = AudioSegment(
    stereo_audio_int16.tobytes(),
    frame_rate=44100,
    sample_width=stereo_audio_int16.dtype.itemsize,
    channels=2
)

# Export as WAV
stereo_audio_segment.export("stereo_output_5.wav", format="wav")

<_io.BufferedRandom name='stereo_output_5.wav'>